### Data Ingestion and Label Mapping
##### This section handles the initial transformation of raw text into a computable format. By mapping categorical labels to binary integers ($0$ for 'Ham', $1$ for 'Spam'), we establish the numerical ground truth required for distance-based calculations.

In [2]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report
from sklearn.feature_selection import f_classif

# Load Dataset/
df = pd.read_csv('/kaggle/input/datasets/organizations/uciml/sms-spam-collection-dataset/spam.csv', encoding='latin-1')
df = df[['v1', 'v2']].rename(columns={'v1': 'label', 'v2': 'text'})
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

print(f"Data Loaded: {len(df)} messages.")

Data Loaded: 5572 messages.


### Fisher’s Linear Discriminant Analysis ($J$)
##### The Fisher Score is a key metric used to evaluate the inherent separability of the dataset by comparing the "spread" within classes to the "distance" between class means.
##### Calculated the global mean and variance for 1,000 top features across both classes using the formula $J = \frac{(\mu_0 - \mu_1)^2}{\sigma_0^2 + \sigma_1^2}$

In [3]:
def calculate_fisher(X, y):
    # Convert sparse matrix to array for math operations
    X_array = X.toarray()
    y_array = y.values
    
    # Split classes
    X_0 = X_array[y_array == 0]
    X_1 = X_array[y_array == 1]
    
    # Mean and Variance per feature (word)
    m0, m1 = X_0.mean(axis=0), X_1.mean(axis=0)
    v0, v1 = X_0.var(axis=0), X_1.var(axis=0)
    
    # Fisher Formula: (m0 - m1)^2 / (v0 + v1)
    # 1e-6 prevents division by zero
    score = (m0 - m1)**2 / (v0 + v1 + 1e-6)
    return np.mean(score)

# Global Fisher Score (J)
global_j = calculate_fisher(TfidfVectorizer(max_features=1000).fit_transform(df['text']), df['label_num'])
print(f"Global Fisher Score (J): {global_j:.5f}")

Global Fisher Score (J): 0.01159


### Interpretation:
##### A Global Fisher Score of 0.01159 is statistically near zero, which leads to several critical conclusions: 
- Extreme Class Overlap: The result proves that the "Ham" and "Spam" vocabularies are mathematically enmeshed. The words used to identify a message as Spam are frequently present in legitimate Ham messages, making them poor "discriminators" when viewed in isolation.
-  Linear Inseparability: Because the score is so low, it is mathematically impossible to draw a straight line (a linear hyperplane) that accurately separates these two classes without high error. This justifies why a simple linear model will always struggle with this specific dataset.
- Variance vs. Distance: The denominator ($\sigma_{ham}^2 + \sigma_{spam}^2$) is significantly larger than the numerator. This means the "noise" or variety of language within each class is much greater than the actual distance between the "average" Ham and "average" Spam message.

### Solutions to Address a Low Fisher Score
##### Given that the linear signal is weak ($J \approx 0.01$), the following interventions are required to improve classification performance:
- Non-Linear Transformation: Deploy Deep Neural Networks with non-linear activation functions (e.g., ReLU). These models "warp" the high-dimensional space to create the separation that does not exist in a flat, linear projection.
- Contextual Feature Modeling (Self-Attention): Utilize Transformer-based architectures (e.g., BERT). By using self-attention mechanisms, these models look at word sequences and semantic context rather than isolated tokens. This effectively increases the discriminative power by identifying specific phrase structures that individual word analysis misses.
- Kernel Projection: Employing Kernel-based methods to project the data into a higher-dimensional Hilbert space, where a hyperplane can more easily separate the overlapping clusters.

### 1. Class Imbalance ($D_{imb}$): 
##### Formula:$$D_{imb} = \log_{10} \left( \frac{N_{majority}}{N_{minority}} \right)$$
##### (Where $N$ represents the number of samples in a given class)

In [4]:
# A. Imbalance (D_imb)
ir = len(df[df['label_num']==0]) / len(df[df['label_num']==1])
d_imb = np.log10(ir)
print(f"Class Imbalance: {d_imb:4f}")

Class Imbalance: 0.810177


### Interpretation: 
##### This value measures the severity of the uneven distribution between the "Ham" and "Spam" classes. A score of ~0.81 is mathematically significant. It warns that standard probabilistic models will inherently develop a bias toward the majority class to minimize overall error. While this yields a deceptively high global accuracy, it actively suppresses the model's ability to learn the minority class. This mathematically explains the "Recall Trap" observed in the baseline model, where the algorithm confidently predicts "Ham" but misses a substantial portion of actual "Spam" messages.

### 2. Dimensionality ($D_{dim}$): The Feature-to-Sample Ratio
##### Formula:$$D_{dim} = \frac{N_{features}}{N_{samples}}$$

In [5]:
# B. Dimensionality (D_dim)
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df['text'])
d_dim = X.shape[1] / X.shape[0]
print(f"Class Dimensionality: {d_dim:4f}")

Class Dimensionality: 0.897344


### Interpretation:
##### This metric quantifies the "Curse of Dimensionality." With a score of ~0.90, the dataset is nearing a critical 1:1 ratio between the number of vectorized words (features) and the number of actual text messages (samples). In such a high-dimensional, sparse matrix, traditional machine learning algorithms struggle to find generalizable patterns because there is too much empty space. This high value alerts us that the model is at extreme risk of overfitting—it might memorize the training data rather than learning true linguistic markers of spam.

### 3. Class Overlap ($D_{over}$): Statistical Vocabulary Entanglement
##### Formula:$$D_{over} = \frac{1}{1 + J}$$
##### (Where $J$ represents the Global Fisher Score: $J = \frac{(\mu_1 - \mu_2)^2}{\sigma_1^2 + \sigma_2^2}$)

In [6]:
# C. Overlap (D_over) - Derived from Fisher
d_over = 1 / (1 + global_j)
print(f"Class Overlap: {d_over:4f}")

Class Overlap: 0.988544


### Interpretation:
##### Derived directly from the dataset's extremely low Fisher Score, this is the most critical stressor in the dataset. A value of ~0.99 indicates near-total statistical overlap between the classes. It proves mathematically that "Spam" messages heavily hijack the vocabulary of normal "Ham" messages. Because the word distributions are practically enmeshed, simple frequency-based filters (like Naive Bayes) lack the discriminatory power to separate the classes. This mandates the use of context-aware models (like LLMs) that can understand word sequence rather than just word presence.

### 4. Class Noise ($D_{noise}$): Lexical Irregularity and Adversarial Characters
##### Formula:$$D_{noise} = \frac{1}{N_{total}} \sum_{i=1}^{N} \left( \frac{\text{Length of Non-Alphabetic Characters}_i}{\text{Total Length of Text}_i} \right)$$

In [7]:
# D. Noise (D_noise)
def get_noise(t):
    return len(re.sub(r'[A-Za-z\s]', '', t)) / len(t) if len(t) > 0 else 0
d_noise = df['text'].apply(get_noise).mean()
print(f"Class Noise: {d_noise:4f}")

Class Noise: 0.081309


### Interpretation:
##### This component measures the prevalence of unstructured linguistic data, such as slang, excessive punctuation, or adversarial misspellings (e.g., "w1nn3r"). At approximately ~0.08 (or 8%), the noise level is relatively low compared to the other stressors. While adversarial characters are present, this score indicates that lexical irregularity is not the primary driver of dataset difficulty. The model's failure points lie in geometry and overlap, rather than in formatting anomalies.

### 5. Class Separability ($D_{sep}$): Geometric Cluster Cohesion
##### Formula:$$D_{sep} = \frac{d_{intra}}{d_{intra} + d_{inter}}$$
##### (Where $d_{intra}$ is the average distance of samples to their class center, and $d_{inter}$ is the distance between the two class centers)

In [8]:
# E. Separability (D_sep) - Centers vs Spread
m0 = X[df['label_num'].values == 0].mean(axis=0)
m1 = X[df['label_num'].values == 1].mean(axis=0)
inter = np.linalg.norm(np.array(m0) - np.array(m1))
intra = np.mean(np.linalg.norm(X[:1000].toarray() - np.array(m0), axis=1)) # Sampled for speed
d_sep = intra / (intra + inter)
print(f"Class Separability: {d_sep:.4f}")

Class Separability: 0.8421


### Interpretation:
##### Separability maps the topological distribution of the dataset in Euclidean space. A score of 0.84 is highly elevated, indicating that the internal chaos or "spread" within the classes ($d_{intra}$) is vastly larger than the physical distance separating the classes ($d_{inter}$). Geometrically, the data points do not form tight, isolated clusters; instead, they exist as diffuse, overlapping clouds. This mathematically invalidates the use of linear classifiers (like standard SVMs or Logistic Regression) and requires the deployment of Deep Neural Networks to warp the space and draw non-linear decision boundaries.

### The Dataset Difficulty Index (DDI):
##### It is a high-level synthetic metric that quantifies the "statistical stress" a dataset places on a classification algorithm. Unlike simple accuracy, which only tells you how a model performed, the DDI tells you why a model might fail before you even begin training.

In [9]:
# Final DDI
ddi = 0.2 * (d_imb + d_dim + d_over + d_noise + d_sep)
print(f"Final DDI: {ddi:.4f}")

Final DDI: 0.7239


### Interpretation:
##### A Final DDI of 0.7239 places this specific dataset in the "Hard" tier. Generally, DDI scores are interpreted through the following thresholds:
- Easy (0.0 – 0.3): Classes are well-separated and balanced.
- Medium (0.3 – 0.6): Some overlap or imbalance exists, but standard models perform reliably.
- Hard (> 0.6): The dataset contains structural conflicts that require advanced architectures.

### Solutions to Solve High DDI Stressors
##### To overcome a DDI of 0.7239, we must move beyond standard classification and implement targeted architectural solutions:

- A. Solving Overlap $(D over)$ with Attention
Standard TF-IDF looks at words in isolation. Large Language Models (LLMs) use Self-Attention to understand context. For example, the word "WINNER" in a "Ham" context (e.g., "Our team was the winner!") is distinguishable from "Spam" context (e.g., "You are a winner! Claim now!") based on surrounding tokens.

- B. Solving Imbalance $(D imb)$ with Focal Loss
Instead of standard Cross-Entropy, use Focal Loss. This mathematically down-weights easy-to-classify "Ham" samples and forces the neural network to focus its "attention" and learning gradient on the rare, difficult "Spam" samples.

- C. Solving Separability $(D sep)$ with Non-Linear Mapping
Use Deep Neural Networks with ReLU activation layers. These layers "bend" and "warp" the high-dimensional feature space, creating artificial separation between classes that the Fisher Score proved does not exist in a flat, linear plane.

- D. Solving Dimensionality $(D dim)$ with Embeddings
Instead of a 5,000-column TF-IDF matrix, use Dense Word Embeddings (like GloVe or BERT embeddings). This compresses the 5,000 sparse features into 768 dense, meaningful dimensions, drastically reducing the $D dim$ stressor and the risk of overfitting.

### Section 4: Empirical Model Validation & Performance Analysis

##### This section executes the final stage of the analytical pipeline: evaluating a baseline linear classifier to empirically test the "statistical stressors" mathematically identified by the Dataset Difficulty Index (DDI).

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, df['label_num'], test_size=0.2, random_state=42)
model = MultinomialNB()
model.fit(X_train, y_train)

print("\n--- CLASSIFICATION REPORT ---")
print(classification_report(y_test, model.predict(X_test), target_names=['Ham', 'Spam']))


--- CLASSIFICATION REPORT ---
              precision    recall  f1-score   support

         Ham       0.97      1.00      0.98       965
        Spam       1.00      0.79      0.88       150

    accuracy                           0.97      1115
   macro avg       0.98      0.89      0.93      1115
weighted avg       0.97      0.97      0.97      1115



### Interpretation:
- While the global Accuracy of 0.97 initially suggests a highly performant model, a granular statistical analysis of the minority class reveals the exact failure points predicted by our theoretical DDI analysis of 0.7239.The Deception of Global Accuracy: In imbalanced datasets ($D_{imb} = 0.81$), global accuracy is an unreliable metric. Because the "Ham" class dominates the sample space, the algorithm achieves 97% accuracy while predominantly acting as a majority-class predictor.
- Spam Precision (1.00): The model exhibits zero False Positives. When the linear boundary flags a message as "Spam," it possesses absolute statistical certainty.
- The "Recall Trap" (Spam Recall: 0.79): This is the critical empirical failure. A recall of $0.79$ indicates the model failed to detect 21% of actual spam messages. This outcome perfectly validates our previous findings: the extreme Class Overlap ($D_{over} = 0.98$) and low Fisher Score ($J = 0.0116$) allow unsolicited messages to statistically "camouflage" themselves within the legitimate vocabulary.

#### Conclusion:
The empirical results definitively validate the DDI categorization of this dataset as "Hard". The model's inability to achieve high recall for the minority class proves that the dataset's overlapping high-dimensional space ($D_{dim} = 0.89$) cannot be resolved by simple linear probabilistic models. Achieving production-level reliability will require deploying non-linear architectures or context-aware embeddings (e.g., Transformers) to capture complex linguistic nuances.

In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score
import warnings

# Silence convergence warnings for cleaner output
warnings.filterwarnings('ignore')

# 1. Initialize the Classifiers
# These models represent different mathematical strategies:
# - Linear (LogReg, SVM): Test the linear separability (Fisher Score)
# - Ensemble (RF, XGBoost): Handle high dimensionality and non-linear boundaries
# - Neural (MLP): Attempt to "warp" the feature space to resolve overlap
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000, solver='lbfgs', random_state=42),
    "SVM (Linear)": LinearSVC(C=1.0, dual=False, random_state=42),
    "MLP Neural Network": MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=100, use_label_encoder=False, eval_metric='logloss', random_state=42),
    "Gradient Boosting": HistGradientBoostingClassifier(max_iter=100, random_state=42)
}

results_summary = []

print("--- STARTING SMS SPAM MULTI-MODEL EVALUATION ---")

# Use the X (TF-IDF) and y (label_num) defined in your previous sections
# X_train, X_test, y_train, y_test were defined in Section 4

for name, clf in classifiers.items():
    print(f"\nTraining {name}...")
    
    # Gradient Boosting requires dense input; others handle sparse matrices
    if name == "Gradient Boosting":
        clf.fit(X_train.toarray(), y_train)
        y_pred = clf.predict(X_test.toarray())
    else:
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
    
    # Calculate performance metrics
    acc = accuracy_score(y_test, y_pred)
    f1_spam = f1_score(y_test, y_pred) # Focus on the minority 'Spam' class
    f1_macro = f1_score(y_test, y_pred, average='macro')
    
    results_summary.append({
        "Model": name,
        "Accuracy": acc,
        "Spam F1": f1_spam,
        "Macro F1": f1_macro
    })
    
    print(f"Results for {name}:")
    print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))

# 3. Final Comparison Table
print("\n" + "="*70)
print(f"{'Model':<25} | {'Accuracy':<10} | {'Spam F1':<10} | {'Macro F1':<10}")
print("-" * 70)
for res in results_summary:
    print(f"{res['Model']:<25} | {res['Accuracy']:<10.4f} | {res['Spam F1']:<10.4f} | {res['Macro F1']:<10.4f}")
print("="*70)

--- STARTING SMS SPAM MULTI-MODEL EVALUATION ---

Training Logistic Regression...
Results for Logistic Regression:
              precision    recall  f1-score   support

         Ham       0.96      1.00      0.98       965
        Spam       1.00      0.73      0.84       150

    accuracy                           0.96      1115
   macro avg       0.98      0.86      0.91      1115
weighted avg       0.96      0.96      0.96      1115


Training SVM (Linear)...
Results for SVM (Linear):
              precision    recall  f1-score   support

         Ham       0.98      1.00      0.99       965
        Spam       0.98      0.87      0.92       150

    accuracy                           0.98      1115
   macro avg       0.98      0.93      0.96      1115
weighted avg       0.98      0.98      0.98      1115


Training MLP Neural Network...
Results for MLP Neural Network:
              precision    recall  f1-score   support

         Ham       0.98      1.00      0.99       965
      

### Interpretation of output:
1. The Linear Paradox: Fisher Score vs SVM
    Our initial analysis identified a Global Fisher Score ($J$) of 0.01159, suggesting extreme linear inseparability. However, Linear SVM achieved a high 98.03% accuracy and a 0.9220 Spam F1.
   - The Interpretation: In high-dimensional text spaces (5000 features), we encounter Cover's Theorem. Even if a dataset is non-separable in a low-dimensional space, projecting it into a high-dimensional sparse matrix (our TF-IDF) makes it much more likely to be linearly separable.
   - The Verdict: The Linear SVM successfully navigated the Dimensionality Stressor ($D_{dim} \approx 0.90$) to find a hyperplane that Logistic Regression missed.
2. Resolving the Overlap ($D_{over}$):
    The Boosting & Neural EdgeXGBoost (0.9236 Spam F1) and the MLP Neural Network (0.9193 Spam F1) represent the upper ceiling of our performance.
   - Navigating the Entanglement: These models are designed to handle the 0.9885 Class Overlap. Where Logistic Regression ($F1 = 0.84$) fell into the "Recall Trap"—failing to distinguish spam that used "ham-like" vocabulary—XGBoost used its tree-based boosting to identify specific word interactions (e.g., "Winner" + "Claim" vs. "Winner" + "Team").
   - The Geometric Warp: The MLP used its hidden layers to physically "bend" the feature space, resolving the Separability Stressor ($D_{sep} = 0.84$) by creating artificial distance between clusters that are overlapping in the raw TF-IDF projection.
3. The "Recall Trap" Breakdown:
   Looking at the Logistic Regression results, we see a Precision of 1.00 but a Recall of 0.73 for Spam.
   - Theoretical Link: This is the Imbalance Index ($D_{imb} = 0.81$) in action. Because the model is overwhelmed by "Ham" samples, its probability threshold is naturally pushed to be conservative.
   - The Impact: It would rather miss 27% of spam messages (False Negatives) than risk mislabeling a single legitimate text as spam. In a "Hard" dataset, linear models often trade off recall for precision to maintain stability.

### Final Conclusion - "Hard"
Even though accuracy is high, the dataset is definitively Hard because:Macro-Metrics are Deceptive: High accuracy masks a significant struggle to resolve the minority class.
- Structural Resistance: There is a persistent ~8% gap in performance ($1.00 - 0.92$) that even advanced ensembles cannot bridge using TF-IDF alone.
- Strategic Recommendation: To push past the 0.92 F1 ceiling, the data suggests we have exhausted the utility of lexical (word-count) features. The next step in this research pipeline is to transition to semantic embeddings (e.g., BERT or RoBERTa).
- By replacing the 5000 sparse TF-IDF features with 768 dense semantic dimensions, you can resolve the Dimensionality ($D_{dim}$) stressor while allowing the model to understand the intent behind the words, effectively "uncoupling" the enmeshed vocabulary identified in our Overlap ($D_{over}$) analysis.